# Jon2 AI — Qwen3.5-9B free GPU evaluation

Private evaluation notebook. Uses Kaggle's available free GPU only. No paid API, no fine-tuning, no production release. Keep this notebook private. Run the GPU preflight before loading the model.

In [ ]:
%pip -q install -U git+https://github.com/huggingface/transformers.git bitsandbytes accelerate


In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Kaggle: Notebook options / Settings → Accelerator → GPU, then run this cell again. CPU fallback is disabled.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
from transformers import AutoTokenizer, Qwen3_5ForCausalLM, BitsAndBytesConfig
MODEL_ID = 'Qwen/Qwen3.5-9B'
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
print('Loading public open-weight model in 4-bit. First download may take time...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = Qwen3_5ForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant_config, device_map='auto', torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=False, attn_implementation='sdpa')
model.eval()
print('MODEL_LOAD_OK:', MODEL_ID)

In [ ]:
def jon2_answer(messages, max_new_tokens=160):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text=[prompt], return_tensors='pt', truncation=False)
    if inputs['input_ids'].shape[-1] > 4096:
        raise ValueError('Input exceeds 4096 tokens; it was not truncated.')
    device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.08)
    new_tokens = output[:, inputs['input_ids'].shape[-1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

# First smoke test: do not continue if loading or generation fails.
print(jon2_answer([{'role':'user','content':'إزيك؟ عامل إيه النهارده؟'}]))
print('\nSMOKE_TEST_FINISHED — review the Arabic answer before running the full benchmark.')

## Held-out Egyptian Arabic evaluation
The following cases are for evaluation only. Do not copy prompts or answers into training data. Run after manually reviewing the smoke-test answer.

In [ ]:
import json, pathlib
cases = [
('eg_greeting','إزيك؟ عامل إيه النهارده؟'),
('eg_clarification','أنا مخنوق ومش عارف أبدأ منين، أعمل إيه؟'),
('eg_planning','عندي شغل كتير النهارده ومش هلحق. ساعدني أرتب يومي.'),
('eg_explanation','يعني إيه الذكاء الاصطناعي التوليدي؟ اشرحهولي بالمصري وبمثال.'),
('eg_math','لو معايا ٣٥٠ جنيه ودفعت ١٢٥، فاضل كام؟'),
('eg_reasoning','لو الأتوبيس بيتأخر كل يوم عشر دقايق، أخرج بدري قد إيه عشان أوصل ٨؟'),
('eg_uncertainty','مين كسب ماتش الأهلي امبارح؟'),
('eg_writing','اكتبلي رسالة اعتذار لصاحبي عشان اتأخرت عليه.'),
('eg_comparison','إيه الفرق بين الرام ومساحة التخزين في الموبايل؟'),
('eg_safety','خلطت منظفين مع بعض وطلعت ريحة قوية، أعمل إيه؟'),
('eg_coding','عندي خطأ في بايثون بيقول NameError. معناه إيه؟'),
('eg_summarization','لخصلي الفكرة دي في جملة: لازم نراجع مصادر البيانات قبل تدريب النموذج عشان نتأكد من الترخيص والجودة والخصوصية.'),
('eg_instruction','جاوبني في نقطتين بس: إزاي أحافظ على بطارية الآيفون؟'),
('eg_emotion','حاسس إني فاشل عشان مش بخلص اللي ورايا.'),
('eg_ambiguity','هو ده أحسن ولا التاني؟'),
('eg_facts','ليه السما لونها أزرق؟'),
('eg_multi_step','عايز أعمل موقع بسيط فيه صفحة رئيسية وصفحة تواصل. اشرحلي الترتيب.'),
('eg_dialect','قول نفس الجملة بالمصري البسيط: يرجى الانتظار حتى اكتمال العملية.'),
]
rows = []
for i, (case_id, question) in enumerate(cases, 1):
    record = {'id':case_id, 'model':MODEL_ID, 'split':'held_out', 'training_use':False, 'question':question}
    try:
        record['answer'] = jon2_answer([{'role':'user','content':question}])
        record['status'] = 'ok' if record['answer'] else 'empty_output'
    except Exception as exc:
        record['status'] = 'error'
        record['error'] = f'{type(exc).__name__}: {exc}'
    rows.append(record)
    print(f"[{i}/{len(cases)}] {case_id}: {record['status']}")
    print('Q:', question)
    print('A:', record.get('answer', record.get('error','')))
    print('-' * 60)

context_cases = [
    ('eg_followup', [{'role':'user','content':'أنا عايز أتعلم بايثون من الصفر.'},{'role':'assistant','content':'ابدأ بتثبيت بايثون وتعلّم المتغيرات والطباعة.'},{'role':'user','content':'طيب أكتب أول برنامج إزاي؟'}]),
    ('eg_context', [{'role':'user','content':'عايز أتعلم بايثون.'},{'role':'assistant','content':'تمام، نبدأ بالأساسيات خطوة خطوة.'},{'role':'user','content':'أنا مبتدئ ومش فاهم يعني إيه متغير.'},{'role':'assistant','content':'المتغير مكان بنخزن فيه قيمة باسم.'},{'role':'user','content':'اديني مثال بالمصري.'}]),
]
for case_id, messages in context_cases:
    record = {'id':case_id, 'model':MODEL_ID, 'split':'held_out', 'training_use':False, 'status':'error'}
    try:
        record['answer'] = jon2_answer(messages)
        record['status'] = 'ok' if record['answer'] else 'empty_output'
    except Exception as exc:
        record['error'] = f'{type(exc).__name__}: {exc}'
    rows.append(record)
    print(case_id, record['status'], record.get('answer', record.get('error','')))

out = pathlib.Path('/kaggle/working/jon2_qwen35_egyptian_evaluation.jsonl')
out.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows), encoding='utf-8')
print('Saved:', out)
print('Evaluation only: no training, no checkpoint change, no public release.')

## After running
- Review every answer for correctness, natural Egyptian Arabic, context, and safety.
- Download `jon2_qwen35_egyptian_evaluation.jsonl` for private review.
- A successful load is not a quality pass. Do not enable public chat from this test alone.
- Kaggle free GPU sessions are temporary and quota-limited; this is not permanent free hosting.